# EDA : préparation des données réglementaires

Pipeline : PDF → texte → nettoyage → chunks (par article / par paragraphe) → fichiers JSONL.

- **Solvency II** : directive 2009/138/CE, découpée **par article**
- **IFRS 17** : règlement (UE) 2021/2036, norme découpée **par paragraphe** (corps, Appendix A, B, C). L'Appendix D (modifications d'autres normes) est exclu pour l'instant.

Exécutez les cellules dans l'ordre.

## 1. Configuration

In [1]:
import os, re, json
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\yolde\RAG\rag-regulatory")
if not Path("data/raw").exists() and PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)

print("Dossier de travail :", Path.cwd())
print("PDF :", [p.name for p in Path("data/raw").glob("*.pdf")])
print("TXT :", [p.name for p in Path("data/text").glob("*.txt")] if Path("data/text").exists() else "aucun")

Dossier de travail : c:\Users\yolde\RAG\rag-regulatory
PDF : ['ifrs17_eu_2021_2036.pdf', 'solvency2_directive_2009_138.pdf']
TXT : ['ifrs17_eu_2021_2036.txt', 'solvency2_directive_2009_138.txt']


## 2. Extraction du texte des PDF
Cette cellule ne fait rien si les fichiers `.txt` existent déjà. Sinon elle a besoin de PyMuPDF (`%pip install pymupdf`, puis Restart du kernel).

In [2]:
RAW, TEXT = Path("data/raw"), Path("data/text")
TEXT.mkdir(parents=True, exist_ok=True)

todo = [p for p in RAW.glob("*.pdf") if not (TEXT / (p.stem + ".txt")).exists()]
if todo:
    import pymupdf
    for pdf_path in todo:
        doc = pymupdf.open(pdf_path)
        text = "\n".join(page.get_text() for page in doc)
        (TEXT / (pdf_path.stem + ".txt")).write_text(text, encoding="utf-8")
        print(f"{pdf_path.name} : {len(doc)} pages, {len(text)} caractères")
else:
    print("Textes déjà extraits, rien à faire.")

Textes déjà extraits, rien à faire.


## 3. Chargement des textes

In [3]:
solv = Path("data/text/solvency2_directive_2009_138.txt").read_text(encoding="utf-8")
ifrs = Path("data/text/ifrs17_eu_2021_2036.txt").read_text(encoding="utf-8")
print(len(solv.split("\n")), "lignes (directive),", len(ifrs.split("\n")), "lignes (IFRS 17)")

18519 lignes (directive), 4213 lignes (IFRS 17)


## 4. Nettoyage
Suppression des espaces insécables (`\xa0`), des traits d'union invisibles (`\xad`) et des lignes parasites répétées à chaque page (« EN », « Official Journal... »).

In [4]:
NOISE = [
    re.compile(r"^EN$"),
    re.compile(r"^Official Journal of the European Union$"),
    re.compile(r"^L \d+/\d+$"),
    re.compile(r"^\d{1,2}\.\d{1,2}\.\d{4}$"),
]

def clean_lines(text):
    text = text.replace("\xa0", " ").replace("\xad\n", "").replace("\xad", "")
    out = []
    for l in text.split("\n"):
        l = l.strip()
        if not l or any(p.match(l) for p in NOISE):
            continue
        out.append(l)
    return out

## 5. Directive Solvency II : un chunk par article
Les articles sont repérés par une ligne `Article N` en suivant la suite 1, 2, 3... dans l'ordre (ce qui élimine les renvois du type « voir l'Article 101 »). Les articles longs sont coupés aux numéros de paragraphe.

In [5]:
def pack_paragraphs(lines, max_chars=3500):
    """Coupe un long article aux numéros de paragraphe ('1.', '2.' seuls sur une ligne)."""
    groups, cur = [], []
    for l in lines:
        if re.fullmatch(r"\d+\.", l) and cur:
            groups.append(cur)
            cur = []
        cur.append(l)
    if cur:
        groups.append(cur)
    parts, buf = [], ""
    for g in groups:
        t = " ".join(g)
        if buf and len(buf) + len(t) > max_chars:
            parts.append(buf)
            buf = t
        else:
            buf = (buf + " " + t).strip()
    if buf:
        parts.append(buf)
    return parts

sl = clean_lines(solv)

cand = [(i, int(l.split()[1])) for i, l in enumerate(sl) if re.fullmatch(r"Article \d+", l)]
headings, expected = [], 1
for i, n in cand:
    if n == expected:
        headings.append((i, n))
        expected += 1

# La directive se termine avant les annexes
last = headings[-1][0]
end_all = next((i for i in range(last, len(sl)) if sl[i] == "ANNEX I"), len(sl))

solv_chunks = []
for k, (start, n) in enumerate(headings):
    end = headings[k + 1][0] if k + 1 < len(headings) else end_all
    title = sl[start + 1]
    body = sl[start + 2 : end]
    parts = pack_paragraphs(body)
    for j, p in enumerate(parts, 1):
        solv_chunks.append({
            "id": f"SII-Art{n}" + (f"-p{j}" if len(parts) > 1 else ""),
            "source": "Directive 2009/138/EC (Solvency II)",
            "article": n,
            "title": title,
            "text": p,
        })

assert len({c["id"] for c in solv_chunks}) == len(solv_chunks), "identifiants en double"
print("Articles retrouvés :", len(headings), "(dernier :", headings[-1][1], ")")
print("Chunks directive :", len(solv_chunks))
print([c["id"] for c in solv_chunks[:8]])

Articles retrouvés : 312 (dernier : 312 )
Chunks directive : 332
['SII-Art1', 'SII-Art2-p1', 'SII-Art2-p2', 'SII-Art3', 'SII-Art4', 'SII-Art5', 'SII-Art6', 'SII-Art7']


## 6. IFRS 17 : un chunk par paragraphe
La norme est découpée en quatre parties : corps (paragraphes `1.`, `2.`...), Appendix A (définitions, regroupées par blocs), Appendix B et C (paragraphes `B1`, `C1`... sans point). L'Appendix D est exclu.

In [6]:
il = clean_lines(ifrs)
il = il[next(i for i, l in enumerate(il) if l == "OBJECTIVE"):]

def find_line(lines, text, start=0):
    return next(i for i in range(start, len(lines)) if lines[i] == text)

iA = find_line(il, "Appendix A")
iB = find_line(il, "Appendix B", iA)
iC = find_line(il, "Appendix C", iB)
iD = find_line(il, "Appendix D", iC)
print("Appendices (lignes) :", iA, iB, iC, iD)

PARA = re.compile(r"^(?:(?:B|C)\d+[A-Z]*|\d+[A-Z]+\.?|\d+\.)$")
SOURCE = "IFRS 17 Insurance Contracts (Regulation (EU) 2021/2036)"

def is_heading(line):
    return (line.isupper() and len(line) >= 8 and len(line.split()) <= 12
            and not PARA.match(line) and not line.startswith("("))

def parse_numbered(lines, part):
    out, section, cur_id, cur_txt = [], part, None, []
    def flush():
        if cur_id and cur_txt:
            out.append({"paragraph": cur_id, "part": part,
                        "section": section, "text": " ".join(cur_txt)})
    for line in lines:
        if PARA.match(line):
            flush()
            cur_id, cur_txt = line.rstrip("."), []
        elif is_heading(line):
            section = line
        else:
            cur_txt.append(line)
    flush()
    return out

def parse_glossary(lines, part, max_chars=2500):
    out, buf = [], ""
    for line in lines:
        if buf and len(buf) + len(line) > max_chars:
            out.append(buf)
            buf = line
        else:
            buf = (buf + " " + line).strip()
    if buf:
        out.append(buf)
    return [{"paragraph": f"AppA-{k}", "part": part, "section": "Defined terms",
             "text": t} for k, t in enumerate(out, 1)]

raw = (parse_numbered(il[:iA], "Body")
       + parse_glossary(il[iA + 2 : iB], "Appendix A")
       + parse_numbered(il[iB:iC], "Appendix B")
       + parse_numbered(il[iC:iD], "Appendix C"))

ifrs_chunks, seen = [], {}
for c in raw:
    base = f"IFRS17-{c['paragraph']}"
    seen[base] = seen.get(base, 0) + 1
    cid = base if seen[base] == 1 else f"{base}-dup{seen[base]}"
    ifrs_chunks.append({"id": cid, "source": SOURCE, **c})

print("Chunks IFRS 17 :", len(ifrs_chunks))
for part in ["Body", "Appendix A", "Appendix B", "Appendix C"]:
    ids = [c["id"] for c in ifrs_chunks if c["part"] == part]
    print(f"{part:<11}", len(ids), ids[:3], "...", ids[-3:])
print("Doublons :", [c["id"] for c in ifrs_chunks if "-dup" in c["id"]])

Appendices (lignes) : 1181 1317 2704 3085
Chunks IFRS 17 : 334
Body        145 ['IFRS17-1', 'IFRS17-2', 'IFRS17-3'] ... ['IFRS17-130', 'IFRS17-131', 'IFRS17-132']
Appendix A  3 ['IFRS17-AppA-1', 'IFRS17-AppA-2', 'IFRS17-AppA-3'] ... ['IFRS17-AppA-1', 'IFRS17-AppA-2', 'IFRS17-AppA-3']
Appendix B  149 ['IFRS17-B1', 'IFRS17-B2', 'IFRS17-B3'] ... ['IFRS17-B135', 'IFRS17-B136', 'IFRS17-B137']
Appendix C  37 ['IFRS17-C1', 'IFRS17-C2', 'IFRS17-C3'] ... ['IFRS17-C32', 'IFRS17-C33', 'IFRS17-C34']
Doublons : []


## 7. Contrôles et sauvegarde

In [7]:
print(next(c for c in solv_chunks if c["article"] == 101)["text"][:600])
for pid in ["32", "B66"]:
    p = next((c for c in ifrs_chunks if c["paragraph"] == pid), None)
    print("\n", pid, "|", p["part"], "|", p["section"], "|",
          p["text"][:400] if p else "introuvable")

print()
for name, ch in [("directive", solv_chunks), ("IFRS 17", ifrs_chunks)]:
    sizes = sorted(len(c["text"]) for c in ch)
    print(name, "min / médiane / max :", sizes[0], sizes[len(sizes) // 2], sizes[-1])

Path("data/chunks").mkdir(parents=True, exist_ok=True)
for name, ch in [("solvency2", solv_chunks), ("ifrs17", ifrs_chunks)]:
    with open(f"data/chunks/{name}.jsonl", "w", encoding="utf-8") as f:
        for c in ch:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("\nSauvegardé dans data/chunks/")

1. The Solvency Capital Requirement shall be calculated in accordance with paragraphs 2 to 5. 2. The Solvency Capital Requirement shall be calculated on the presumption that the undertaking will pursue its business as a going concern. 3. The Solvency Capital Requirement shall be calibrated so as to ensure that all quantifiable risks to which an insurance or reinsurance undertaking is exposed are taken into account. It shall cover existing business, as well as the new business expected to be written over the following 12 months. With respect to existing business, it shall cover only unexpected 

 32 | Body | MEASUREMENT (PARAGRAPHS B36–B119F) | On initial recognition, an entity shall measure a group of insurance contracts at the total of: (a) the fulfilment cash flows, which comprise: (i) estimates of future cash flows (paragraphs 33–35); (ii) an adjustment to reflect the time value of money and the financial risks related to the future cash flows, to the extent that the financial risks

In [8]:
def split_long(text, max_chars=2000):
    if len(text) <= max_chars:
        return [text]
    sents = re.split(r"(?<=[.;:])\s+", text)
    pieces = []
    for s in sents:                      # une phrase trop longue est coupée aux espaces
        while len(s) > max_chars:
            cut = s.rfind(" ", 0, max_chars)
            cut = cut if cut > 0 else max_chars
            pieces.append(s[:cut])
            s = s[cut:].strip()
        pieces.append(s)
    parts, buf = [], ""
    for s in pieces:
        if buf and len(buf) + len(s) + 1 > max_chars:
            parts.append(buf)
            buf = s
        else:
            buf = (buf + " " + s).strip()
    if buf:
        parts.append(buf)
    return parts

def finalize(chunks, max_chars=2000, min_chars=30):
    out = []
    for c in chunks:
        t = c["text"].strip()
        if len(t) < min_chars or "[deleted]" in t.lower():
            continue
        parts = split_long(t, max_chars)
        for j, p in enumerate(parts, 1):
            d = dict(c)
            d["text"] = p
            if len(parts) > 1:
                d["id"] = f"{c['id']}-s{j}"
            out.append(d)
    return out

solv_final = finalize(solv_chunks)
ifrs_final = finalize(ifrs_chunks)

for name, ch in [("directive", solv_final), ("IFRS 17", ifrs_final)]:
    assert len({c["id"] for c in ch}) == len(ch), "identifiants en double"
    sizes = sorted(len(c["text"]) for c in ch)
    print(f"{name}: {len(ch)} chunks | min/médiane/max : {sizes[0]} / {sizes[len(sizes)//2]} / {sizes[-1]}")

removed = [c["id"] for c in ifrs_chunks if len(c["text"]) < 30 or "[deleted]" in c["text"].lower()]
print("IFRS 17 retirés (trop courts ou supprimés) :", removed)

for name, ch in [("solvency2", solv_final), ("ifrs17", ifrs_final)]:
    with open(f"data/chunks/{name}_final.jsonl", "w", encoding="utf-8") as f:
        for c in ch:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")
print("Sauvegardé : data/chunks/*_final.jsonl")

directive: 408 chunks | min/médiane/max : 46 / 965 / 2000
IFRS 17: 348 chunks | min/médiane/max : 66 / 531 / 1991
IFRS 17 retirés (trop courts ou supprimés) : ['IFRS17-27']
Sauvegardé : data/chunks/*_final.jsonl


In [9]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import sys
print(sys.executable)
!{sys.executable} -m pip install sentence-transformers

c:\Users\yolde\.venv\Scripts\python.exe



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import sys, subprocess
r = subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], capture_output=True, text=True)
print(r.returncode)
print(r.stdout[-800:])
print(r.stderr[-800:])

0
Looking in links: c:\Users\yolde\AppData\Local\Temp\tmp1wbdu705




In [12]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers"],
                   capture_output=True, text=True)
print(r.returncode)
print(r.stdout[-800:])
print(r.stderr[-800:])

0
es (from typer->transformers<6.0.0,>=5.0.0->sentence-transformers) (15.0.0)


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



In [13]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

chunks = []
for name in ["solvency2_final", "ifrs17_final"]:
    with open(f"data/chunks/{name}.jsonl", encoding="utf-8") as f:
        chunks += [json.loads(l) for l in f]
print("Chunks chargés :", len(chunks))

def label(c):
    if "article" in c:
        return f"{c['source']} - Article {c['article']} - {c['title']}"
    return f"{c['source']} - paragraph {c['paragraph']} - {c['section']}"

model = SentenceTransformer("intfloat/multilingual-e5-base")
model.max_seq_length = 512

texts = ["passage: " + label(c) + "\n" + c["text"] for c in chunks]
emb = model.encode(texts, batch_size=16, normalize_embeddings=True, show_progress_bar=True)

np.save("data/chunks/embeddings.npy", emb)
print("Matrice d'embeddings :", emb.shape)

Chunks chargés : 756


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/48 [00:00<?, ?it/s]

Matrice d'embeddings : (756, 768)


In [14]:
def search(question, k=5):
    q = model.encode(["query: " + question], normalize_embeddings=True)[0]
    scores = emb @ q
    for i in np.argsort(scores)[::-1][:k]:
        c = chunks[i]
        ref = f"Art. {c['article']}" if "article" in c else f"IFRS 17 §{c['paragraph']}"
        print(f"{scores[i]:.3f} | {c['id']} | {ref}")
        print("   ", c["text"][:200].replace("\n", " "), "...\n")

search("What is the Solvency Capital Requirement and what confidence level does it use?")
print("=" * 60)
search("Qu'est-ce que la marge de service contractuelle ?")
print("=" * 60)
search("risk adjustment for non-financial risk")

0.870 | SII-Art104-s2 | Art. 104
    Such parameters shall be calibrated on the basis of the internal data of the undertaking concerned, or of data which is directly relevant for the operations of that undertaking using standardised meth ...

0.870 | SII-Art51-s1 | Art. 51
    1. Member States shall, taking into account the information required in paragraph 3 and the principles set out in paragraph 4 of Article 35, require insurance and reinsurance undertakings to disclose  ...

0.870 | SII-Art105-p2-s2 | Art. 105
    (c) the sensitivity of the values of assets, liabilities and financial instruments to changes in the level or in the volatility of market prices of real estate (property risk); (d) the sensitivity of  ...

0.869 | SII-Art101 | Art. 101
    1. The Solvency Capital Requirement shall be calculated in accordance with paragraphs 2 to 5. 2. The Solvency Capital Requirement shall be calculated on the presumption that the undertaking will pursu ...

0.868 | SII-Art100 | Art. 100


In [15]:
import re, math
from collections import Counter

def tok(s):
    s = re.sub(r"(\d),(\d)", r"\1.\2", s.lower())   # 99,5 -> 99.5
    return re.findall(r"[a-z0-9]+(?:\.[0-9]+)?", s)

docs_tok = [tok(label(c) + " " + c["text"]) for c in chunks]
N = len(docs_tok)
avgdl = sum(len(d) for d in docs_tok) / N
df = Counter(t for d in docs_tok for t in set(d))
idf = {t: math.log(1 + (N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
tfs = [Counter(d) for d in docs_tok]

def bm25_scores(question, k1=1.5, b=0.75):
    out = np.zeros(N)
    for i, (tf, d) in enumerate(zip(tfs, docs_tok)):
        s = 0.0
        for t in tok(question):
            if t in tf:
                f = tf[t]
                s += idf[t] * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
        out[i] = s
    return out

print("BM25 prêt :", N, "chunks")

BM25 prêt : 756 chunks


In [16]:
def hybrid(question, k=5, pool=30, c=60):
    q = model.encode(["query: " + question], normalize_embeddings=True)[0]
    dense, sparse = emb @ q, bm25_scores(question)
    top_d = list(np.argsort(dense)[::-1][:pool])
    top_s = list(np.argsort(sparse)[::-1][:pool])

    fused = {}
    for rank, i in enumerate(top_d):
        fused[i] = fused.get(i, 0) + 1 / (c + rank + 1)
    for rank, i in enumerate(top_s):
        fused[i] = fused.get(i, 0) + 1 / (c + rank + 1)

    for i in sorted(fused, key=fused.get, reverse=True)[:k]:
        ch = chunks[i]
        ref = f"Art. {ch['article']}" if "article" in ch else f"IFRS 17 §{ch['paragraph']}"
        src = ("embeddings" if i in top_d else "") + (" + mots exacts" if i in top_s else "")
        print(f"{ch['id']} | {ref} | trouvé par : {src.strip(' +')}")
        print("   ", ch["text"][:200].replace("\n", " "), "...\n")

hybrid("What is the Solvency Capital Requirement and what confidence level does it use?")
print("=" * 60)
hybrid("Article 101")
print("=" * 60)
hybrid("IFRS 17 paragraph B66")

SII-Art101 | Art. 101 | trouvé par : embeddings + mots exacts
    1. The Solvency Capital Requirement shall be calculated in accordance with paragraphs 2 to 5. 2. The Solvency Capital Requirement shall be calculated on the presumption that the undertaking will pursu ...

SII-Art122 | Art. 122 | trouvé par : embeddings + mots exacts
    1. Insurance and reinsurance undertakings may use a different time period or risk measure than that set out in Article 101(3) for internal modelling purposes as long as the outputs of the internal mod ...

SII-Art230 | Art. 230 | trouvé par : embeddings + mots exacts
    consolidation-based method 1. The calculation of the group solvency of the participating insurance or reinsurance undertaking shall be carried out on the basis of the consolidated accounts. The group  ...

SII-Art238-s1 | Art. 238 | trouvé par : embeddings + mots exacts
    determination of the Solvency Capital Requirement 1. Without prejudice to Article  231, the Solvency Capital Requir

In [17]:
by_art, by_par = {}, {}
for i, c in enumerate(chunks):
    if "article" in c:
        by_art.setdefault(c["article"], []).append(i)
    else:
        by_par.setdefault(c["paragraph"].upper(), []).append(i)

def find_references(question):
    idx = []
    for m in re.finditer(r"\b(?:article|art\.?)\s*(\d+)", question, re.I):
        idx += by_art.get(int(m.group(1)), [])
    for m in re.finditer(r"(?:paragraphe?|para\.?|§)\s*([BC]?\d+[A-Z]?)\b", question, re.I):
        idx += by_par.get(m.group(1).upper(), [])
    for m in re.finditer(r"\b([BC]\d+[A-Z]?)\b", question):   # "B66" écrit seul
        idx += by_par.get(m.group(1), [])
    return list(dict.fromkeys(idx))

print(find_references("Article 101"))
print(find_references("IFRS 17 paragraph B66"))

[131]
[633]


In [18]:
def retrieve(question, k=5, pool=30, c=60):
    q = model.encode(["query: " + question], normalize_embeddings=True)[0]
    dense, sparse = emb @ q, bm25_scores(question)
    top_d = list(np.argsort(dense)[::-1][:pool])
    top_s = list(np.argsort(sparse)[::-1][:pool])
    fused = {}
    for rank, i in enumerate(top_d):
        fused[i] = fused.get(i, 0) + 1 / (c + rank + 1)
    for rank, i in enumerate(top_s):
        fused[i] = fused.get(i, 0) + 1 / (c + rank + 1)
    refs = find_references(question)
    ranked = refs + [i for i in sorted(fused, key=fused.get, reverse=True) if i not in refs]
    return ranked[:k], set(refs)

def show(question, k=5):
    ids, refs = retrieve(question, k)
    for i in ids:
        ch = chunks[i]
        ref = f"Art. {ch['article']}" if "article" in ch else f"IFRS 17 §{ch['paragraph']}"
        tag = "RÉFÉRENCE EXACTE" if i in refs else "recherche"
        print(f"{ch['id']} | {ref} | {tag}")
        print("   ", ch["text"][:200].replace("\n", " "), "...\n")

show("Article 101")
print("=" * 60)
show("IFRS 17 paragraph B66")
print("=" * 60)
show("Quel niveau de confiance exige l'article 101 ?")
print("=" * 60)
show("What does paragraph 32 of IFRS 17 say?")

SII-Art101 | Art. 101 | RÉFÉRENCE EXACTE
    1. The Solvency Capital Requirement shall be calculated in accordance with paragraphs 2 to 5. 2. The Solvency Capital Requirement shall be calculated on the presumption that the undertaking will pursu ...

SII-Art94 | Art. 94 | recherche
    1. Basic own-fund items shall be classified in Tier 1 where they substantially possess the characteristics set out in Article 93(1)(a) and  (b), taking into consideration the features set out in Artic ...

SII-Art301 | Art. 301 | recherche
    1. The Commission shall be assisted by the European Insurance and Occupational Pensions Committee. N E 9 0 0 2.2 1.7 1 2. Where reference is made to this paragraph, Articles 5 and 7 of Decision 1999/4 ...

SII-Art151 | Art. 151 | recherche
    The host Member State shall require the non-life insurance undertaking to ensure that persons pursuing claims arising out of events occurring in its territory are not placed in a less favourable situa ...

SII-Art122 | Art. 1

In [20]:
NOISE = [
    re.compile(r"^EN$"),
    re.compile(r"^Official Journal of the European Union$"),
    re.compile(r"^L \d+/\d+$"),
    re.compile(r"^\d{1,2}\.\d{1,2}\.\d{4}$"),
]
MARGIN_TOKEN = re.compile(r"^(?:[0-9/LNE]|\d\.\d)$")

def drop_margin_text(lines, max_run=14):
    """Supprime l'en-tête de page imprimé à l'envers (N, E, 9, 0, 0, 2.2, ...)."""
    out, i = [], 0
    while i < len(lines):
        if lines[i] == "N" and i + 1 < len(lines) and lines[i + 1] == "E":
            j = i + 2
            while j < len(lines) and j - i <= max_run and MARGIN_TOKEN.match(lines[j]):
                j += 1
            i = j
        else:
            out.append(lines[i])
            i += 1
    return out

def clean_lines(text):
    text = text.replace("\xa0", " ").replace("\xad\n", "").replace("\xad", "")
    out = []
    for l in text.split("\n"):
        l = l.strip()
        if not l or any(p.match(l) for p in NOISE):
            continue
        out.append(l)
    return drop_margin_text(out)